# Metadata-Only LLM + RAG Call-Centre Pipeline
**Author:** Lineo Nkoebe  
**Purpose:** Computational implementation accompanying the MSc Computing dissertation.

This notebook is designed to be understandable, reproducible, and auditable. Please read the repository `README.md` and `docs/TECHNICAL_DEFENCE.md` alongside the code.

## Restricted dataset
The original operational call-centre dataset is intentionally **not distributed through GitHub**. An authorised user who has received the research dataset through an approved channel should place `CallCenterData.csv` in the same folder as this notebook. If it is absent, the notebook will stop at the data-loading cell with an explanatory message.

This notebook is designed to run **top-to-bottom, cell by cell**. The primary path uses **Sentence-Transformers MiniLM** for semantic embeddings, **FAISS cosine-similarity retrieval**, and **FLAN-T5** for constrained generation. If optional model dependencies are unavailable, the notebook enters a clearly labelled **offline validation fallback** so that control flow can be tested. The fallback must not be interpreted as the dissertation model or its performance.


In [ ]:
# Cell 1 — Reproducibility configuration and core imports
from pathlib import Path
import sys, re, math, hashlib, platform, importlib.util, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

AUTHOR = "Lineo Nkoebe"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

BASE_DIR = Path.cwd()
RAW_CSV = BASE_DIR / "CallCenterData.csv"

print(f"Author: {AUTHOR}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {BASE_DIR}")
print(f"Dataset expected at: {RAW_CSV}")

In [ ]:
# Cell 2 — Dissertation-model dependency check
HAS_SENTENCE_TRANSFORMERS = importlib.util.find_spec("sentence_transformers") is not None
HAS_TRANSFORMERS = importlib.util.find_spec("transformers") is not None
HAS_TORCH = importlib.util.find_spec("torch") is not None
HAS_FAISS = importlib.util.find_spec("faiss") is not None

PRIMARY_MODE_AVAILABLE = HAS_SENTENCE_TRANSFORMERS and HAS_TRANSFORMERS and HAS_TORCH and HAS_FAISS
EXECUTION_MODE = "PRIMARY_MINILM_FAISS_FLAN_T5" if PRIMARY_MODE_AVAILABLE else "OFFLINE_VALIDATION_FALLBACK"

print("sentence-transformers available:", HAS_SENTENCE_TRANSFORMERS)
print("transformers available:", HAS_TRANSFORMERS)
print("torch available:", HAS_TORCH)
print("faiss available:", HAS_FAISS)
print("Execution mode:", EXECUTION_MODE)

if not PRIMARY_MODE_AVAILABLE:
    print()
    print("NOTE: One or more primary-model dependencies are unavailable in this environment.")
    print("The notebook will use TF-IDF + scikit-learn cosine retrieval + deterministic fallback generation for execution validation.")
    print("This fallback is NOT presented as the dissertation's MiniLM/FAISS/FLAN-T5 result.")


In [ ]:
# Cell 3 — Load the restricted research dataset using a portable relative path
if not RAW_CSV.exists():
    raise FileNotFoundError(
        "CallCenterData.csv is not distributed with this GitHub repository because the operational research dataset is restricted. "
        "Authorised users who have received the dataset through an approved channel should place CallCenterData.csv "
        "in the repository root beside this notebook, then rerun from Cell 1."
    )

df = pd.read_csv(RAW_CSV, dtype=str)
df.columns = [c.strip() for c in df.columns]
print("Loaded dataframe shape:", df.shape)
print("Columns:", list(df.columns))
print("Raw record previews are intentionally suppressed in the GitHub copy.")


In [ ]:
# Cell 4 — Utility functions

def parse_time_to_seconds(value):
    """Convert values such as '1m 23s', '83s', '2m', or numeric text to seconds."""
    if value is None or pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value) if np.isfinite(value) else np.nan
    s = str(value).strip().lower()
    if not s:
        return np.nan
    total = 0
    found = False
    h = re.search(r"(\d+(?:\.\d+)?)\s*h", s)
    m = re.search(r"(\d+(?:\.\d+)?)\s*m", s)
    sec = re.search(r"(\d+(?:\.\d+)?)\s*s", s)
    if h:
        total += float(h.group(1)) * 3600; found = True
    if m:
        total += float(m.group(1)) * 60; found = True
    if sec:
        total += float(sec.group(1)); found = True
    if found:
        return total
    try:
        return float(s)
    except ValueError:
        return np.nan


def anonymize_identifier(value, salt="msc-metadata-research"):
    """One-way SHA-256 hash for identifiers. Raw identifiers are not used in prompts."""
    if value is None or pd.isna(value):
        return None
    cleaned = re.sub(r"\D", "", str(value))
    if not cleaned:
        return None
    return hashlib.sha256((salt + cleaned).encode("utf-8")).hexdigest()

In [ ]:
# Cell 5 — Cleaning and metadata feature engineering

df_clean = df.copy()

# Convert duration-like fields to seconds.
time_fields = [
    "Hold time", "Duration", "IVR time", "Wait time", "Handle time",
    "Talk time", "Speed to answer", "Time to answer"
]
for col in time_fields:
    if col in df_clean.columns:
        df_clean[f"{col}_sec"] = df_clean[col].apply(parse_time_to_seconds)

# Correctly parse the timestamp column (case-insensitive source name handled by stripped columns).
if "Time period" in df_clean.columns:
    df_clean["time_period_ts"] = pd.to_datetime(df_clean["Time period"], errors="coerce")
    df_clean["hour"] = df_clean["time_period_ts"].dt.hour
    df_clean["weekday"] = df_clean["time_period_ts"].dt.day_name()
    df_clean["is_weekend"] = df_clean["time_period_ts"].dt.dayofweek.isin([5, 6]).astype("Int64")
else:
    raise KeyError("Expected source column 'Time period' was not found.")

# Normalize useful categorical metadata.
cat_fields = [
    "Type", "Direction", "Short abandon", "Business hours", "Service Level Breached",
    "Tag", "Queues Name", "Abandonment reason", "Transferred"
]
for col in cat_fields:
    if col in df_clean.columns:
        df_clean[f"{col}_norm"] = (
            df_clean[col].fillna("unknown").astype(str).str.strip().str.lower()
            .str.replace(r"\s+", "_", regex=True)
        )

# Hash identifiers for privacy; raw identifiers are excluded from model prompts.
for col in ["Number", "Customer Number"]:
    if col in df_clean.columns:
        df_clean[f"{col}_hashed"] = df_clean[col].apply(anonymize_identifier)

print("Cleaned shape:", df_clean.shape)
print("Valid timestamps:", int(df_clean["time_period_ts"].notna().sum()))
print("Timestamp range:", df_clean["time_period_ts"].min(), "to", df_clean["time_period_ts"].max())

In [ ]:
# Cell 6 — Data-quality and descriptive checks
print("Missingness for selected analytical fields (%):")
selected = [c for c in ["Type","Direction","Time period","Queues Name","Wait time","Hold time","Handle time"] if c in df_clean.columns]
missing_pct = (df_clean[selected].isna().mean() * 100).round(2).sort_values(ascending=False)
display(missing_pct.to_frame("missing_%"))

if "Type" in df_clean.columns:
    print()
    print("Call type counts:")
    display(df_clean["Type"].value_counts(dropna=False).to_frame("count"))

for metric in ["Hold time_sec", "Wait time_sec", "Handle time_sec"]:
    if metric in df_clean.columns:
        s = pd.to_numeric(df_clean[metric], errors="coerce").dropna()
        print(f"{metric}: n={len(s):,}, mean={s.mean():.2f}, median={s.median():.2f}, p90={s.quantile(.90):.2f}, p99={s.quantile(.99):.2f}, max={s.max():.2f}")

In [ ]:
# Cell 7 — EDA: Call Type Distribution (original dissertation figure)
# This cell intentionally preserves the original plotting specification used in the study.

if 'Type' in df_clean.columns:
    print("\nCall Type distribution (top 10):")
    display(df_clean['Type'].value_counts(dropna=False).head(10))
    vc = df_clean['Type'].value_counts(dropna=False)
    fig, ax = plt.subplots(figsize=(8,3))
    vc.plot(kind='bar', ax=ax)
    ax.set_title('Call Type Distribution'); ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
else:
    print("No 'Type' column — skipping type distribution chart.")


In [ ]:
# Cell 8 — EDA: Hold-time histogram (original dissertation figure)
# The source data contains hold-time values that are overwhelmingly zero.
# The graph is retained exactly because it faithfully reflects the observed source data.

hold_col = next((c for c in df_clean.columns if 'hold' in c.lower() and c.endswith('_sec')), None)
if hold_col:
    arr = df_clean[hold_col].dropna().astype(float)
    if len(arr) > 0:
        clip = np.percentile(arr, 99)
        fig, ax = plt.subplots(figsize=(8,3))
        ax.hist(arr.clip(upper=clip), bins=50)
        ax.set_title('Hold time (sec) histogram (clipped at 99th %ile)')
        ax.set_xlabel('Seconds'); ax.set_ylabel('Frequency'); plt.tight_layout(); plt.show()
    else:
        print("Hold column exists but contains no numeric values.")
else:
    print("No hold-time column detected.")


In [ ]:
# Cell 9 — EDA: calls by hour of day
calls_by_hour = df_clean.groupby("hour", dropna=True).size().reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4))
calls_by_hour.plot(kind="bar", ax=ax)
ax.set_title("Calls by Hour of Day")
ax.set_xlabel("Hour (0–23)")
ax.set_ylabel("Number of calls")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Build privacy-preserving metadata documents

def row_to_metadata_text(row):
    fields = [
        ("type", "Type_norm"),
        ("direction", "Direction_norm"),
        ("queue", "Queues Name_norm"),
        ("wait_seconds", "Wait time_sec"),
        ("hold_seconds", "Hold time_sec"),
        ("handle_seconds", "Handle time_sec"),
        ("abandonment_reason", "Abandonment reason_norm"),
        ("hour", "hour"),
        ("business_hours", "Business hours_norm"),
    ]
    parts = []
    for label, col in fields:
        if col in row.index and pd.notna(row[col]):
            val = str(row[col]).strip()
            if val and val.lower() not in {"nan", "none"}:
                parts.append(f"{label}: {val}")
    return " | ".join(parts) if parts else "empty metadata record"

# Use a reproducible random sample rather than the first N chronological rows.
SAMPLE_SIZE = min(10000, len(df_clean))
df_for_prompts = df_clean.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
df_for_prompts["metadata_prompt"] = df_for_prompts.apply(row_to_metadata_text, axis=1)
texts = df_for_prompts["metadata_prompt"].tolist()

print("Documents prepared:", len(texts))
print("Individual metadata-record previews are intentionally suppressed in the GitHub copy.")

In [ ]:
# Cell 11 — Semantic embeddings and FAISS retrieval index
# PRIMARY: all-MiniLM-L6-v2 embeddings + FAISS IndexFlatIP (cosine similarity on normalised vectors).
# FALLBACK: TF-IDF + scikit-learn cosine NearestNeighbors for offline execution validation only.

if PRIMARY_MODE_AVAILABLE:
    import faiss
    from sentence_transformers import SentenceTransformer

    EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embs = embedder.encode(
        texts,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    embs = np.ascontiguousarray(np.asarray(embs, dtype=np.float32))

    # Because the embeddings are L2-normalised, inner product is equivalent to cosine similarity.
    faiss_index = faiss.IndexFlatIP(embs.shape[1])
    faiss_index.add(embs)
    RETRIEVAL_BACKEND = f"FAISS IndexFlatIP + {EMBEDDING_MODEL_NAME}"

    def _search_index(query, top_k):
        q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        q = np.ascontiguousarray(np.asarray(q, dtype=np.float32))
        scores, indices = faiss_index.search(q, top_k)
        return scores[0], indices[0]
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.neighbors import NearestNeighbors

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    embs = vectorizer.fit_transform(texts)
    nn_index = NearestNeighbors(metric="cosine", algorithm="brute")
    nn_index.fit(embs)
    RETRIEVAL_BACKEND = "TF-IDF + scikit-learn cosine fallback (offline validation only)"

    def _search_index(query, top_k):
        q = vectorizer.transform([query])
        distances, indices = nn_index.kneighbors(q, n_neighbors=top_k, return_distance=True)
        return 1.0 - distances[0], indices[0]

print("Retrieval backend:", RETRIEVAL_BACKEND)
print("Indexed documents:", len(texts))


def retrieve_inmemory(query, top_k=5):
    if top_k < 1:
        raise ValueError("top_k must be >= 1")
    k = min(top_k, len(texts))
    scores, indices = _search_index(query, k)
    results = []
    for score, idx in zip(scores, indices):
        idx = int(idx)
        if idx < 0:
            continue
        results.append({
            "id": idx,
            "score": float(score),
            "text": texts[idx],
            "source_type": str(df_for_prompts.loc[idx, "Type"])
        })
    return results


In [ ]:
# Cell 12 — Retrieval sanity check
retrieval_queries = [
    "abandoned call after a long queue wait",
    "completed incoming customer call",
    "no-answer outgoing call"
]
for query in retrieval_queries:
    print()
    print("QUERY:", query)
    for item in retrieve_inmemory(query, top_k=3):
        print(f"score={item['score']:.4f} | source_type={item['source_type']} | {item['text'][:180]}")

In [ ]:
# Cell 13 — FLAN-T5 dispatcher with transparent offline fallback
ALLOWED_TAGS = [
    "billing", "registration", "technical_support", "cancellation",
    "new_student_registration", "general_query", "dropped_call_ringing", "unknown"
]

if PRIMARY_MODE_AVAILABLE:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    import torch
    MODEL_NAME = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    llm_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    llm_model.eval()
    GENERATION_BACKEND = MODEL_NAME

    def llm_dispatch_generate(prompt, max_tokens=80):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            output_ids = llm_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
                num_beams=1
            )
        return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
else:
    GENERATION_BACKEND = "deterministic fallback (offline validation only)"

    def llm_dispatch_generate(prompt, max_tokens=80):
        # Deterministic control-flow validator only; this is NOT a substitute for FLAN-T5.
        matches = re.findall(r"(?im)^Question:\s*(.+)$", str(prompt))
        question = matches[-1].strip().lower() if matches else str(prompt).lower()
        if "abandon" in question or "no-answer" in question or "no answer" in question or "dropped" in question:
            return "dropped_call_ringing | Retry contact and route to the appropriate service queue."
        if "registration" in question or "register" in question:
            return "registration | Route to Registration for assistance."
        if "bill" in question or "invoice" in question:
            return "billing | Route to Billing and review the account query."
        if "technical" in question or "password" in question or "login" in question:
            return "technical_support | Route to Technical Support."
        if "cancel" in question:
            return "cancellation | Route to the cancellation process and verify the request."
        return "general_query | Route to a human agent for general assistance."

print("Generation backend:", GENERATION_BACKEND)

In [ ]:
# Cell 14 — Strict TAG | ACTION parser

def normalize_tag(raw_text):
    text = str(raw_text).lower().strip()
    # Exact/word-boundary matching, longer tags checked first to avoid partial collisions.
    for tag in sorted(ALLOWED_TAGS, key=len, reverse=True):
        if re.search(rf"(?<![a-z0-9_]){re.escape(tag)}(?![a-z0-9_])", text):
            return tag
    return "unknown"


def parse_tag_action(output_text):
    collapsed = " ".join(str(output_text).split())
    if "|" in collapsed:
        tag_part, action_part = collapsed.split("|", 1)
        tag = normalize_tag(tag_part)
        action = action_part.strip() or "Escalate to human agent."
        return tag, action, collapsed
    tag = normalize_tag(collapsed)
    return tag, "Escalate to human agent.", collapsed

# Parser unit checks
parser_tests = [
    "registration | Route to Registration.",
    "dropped_call_ringing | Retry contact.",
    "unexpected free text"
]
for test in parser_tests:
    print(test, "->", parse_tag_action(test)[:2])

In [ ]:
# Cell 15 — RAG orchestration
FEW_SHOT = """Examples:
Context: type: abandoned | wait_seconds: 240 | queue: registration
Question: Customer abandoned after a long wait in registration.
Answer: dropped_call_ringing | Retry contact and route to Registration.

Context: type: completed | queue: billing
Question: Customer has an invoice discrepancy.
Answer: billing | Route to Billing and review the account query.
"""


def rag_generate(query, top_k=3, max_tokens=80):
    retrieved = retrieve_inmemory(query, top_k=top_k)
    context = "\n".join(item["text"] for item in retrieved)
    prompt = f"""{FEW_SHOT}
Retrieved metadata context:
{context}

Question: {query}
Return EXACTLY one allowed TAG and one short ACTION in this format:
TAG | ACTION
Allowed TAGs: {', '.join(ALLOWED_TAGS)}
If uncertain: unknown | Escalate to human agent.
Answer:"""
    raw = llm_dispatch_generate(prompt, max_tokens=max_tokens)
    tag, action, collapsed = parse_tag_action(raw)
    return {"query": query, "tag": tag, "action": action, "raw": raw, "retrieved": retrieved}

In [ ]:
# Cell 16 — End-to-end RAG demonstrations
# These are demonstrations of pipeline behaviour, not a substitute for a labelled accuracy study.
demo_queries = [
    "Customer abandoned after a long queue wait and needs a callback.",
    "Customer needs help with registration.",
    "Customer has a billing or invoice query.",
    "Customer cannot log in and needs technical support."
]

demo_results = []
for q in demo_queries:
    result = rag_generate(q, top_k=3)
    demo_results.append(result)
    print()
    print("QUERY:", q)
    print("TAG:", result["tag"])
    print("ACTION:", result["action"])
    print("RAW:", result["raw"])
    print("Top retrieved source types:", [r["source_type"] for r in result["retrieved"]])

In [ ]:
# Cell 17 — Retrieval consistency diagnostic
# This checks whether metadata retrieval returns records whose source Type aligns with simple type-oriented queries.
# It is a diagnostic, not a full model-performance metric.
checks = [
    ("abandoned call with long wait", "abandoned"),
    ("completed customer call", "completed"),
    ("no-answer call", "no-answer"),
]
rows = []
for query, expected_type in checks:
    retrieved = retrieve_inmemory(query, top_k=10)
    matches = sum(str(r["source_type"]).lower() == expected_type for r in retrieved)
    rows.append({"query": query, "expected_type": expected_type, "top10_matching_type": matches, "top10_rate": matches / 10})
retrieval_diagnostic = pd.DataFrame(rows)
display(retrieval_diagnostic)

In [ ]:
# Cell 18 — Final reproducibility summary
summary = {
    "author": AUTHOR,
    "rows_loaded": int(len(df_clean)),
    "columns_loaded": int(df.shape[1]),
    "timestamp_min": str(df_clean["time_period_ts"].min()),
    "timestamp_max": str(df_clean["time_period_ts"].max()),
    "sample_documents_indexed": int(len(texts)),
    "retrieval_backend": RETRIEVAL_BACKEND,
    "generation_backend": GENERATION_BACKEND,
    "execution_mode": EXECUTION_MODE,
}
display(pd.Series(summary, name="value").to_frame())
print()
print("PIPELINE EXECUTION COMPLETE")
print("All notebook cells executed in sequence without hidden state.")
if EXECUTION_MODE != "PRIMARY_MINILM_FAISS_FLAN_T5":
    print("IMPORTANT: This run validated the offline fallback. Re-run with requirements installed to execute MiniLM + FAISS + FLAN-T5.")